# Spark Exercises 07 — UDFs, Built-ins & Performance

This chapter is about what Spark **lets you do** and what it **punishes you for**. You *can* write a plain Python UDF — but it becomes a black box that Catalyst cannot optimize and that serializes every row to Python. You'll see the native alternative, the vectorized `pandas_udf` middle ground, and the partition controls (`repartition` / `coalesce` / `cache`) that decide how fast a job runs on 250,000 rows.

**Data files:** `data/web_events.csv` (250k rows), `data/orders.csv`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read `data/web_events.csv` (250,000 rows) into `web` (`header=True`, `inferSchema=True`) and `count()` it.

### 3. How many **partitions** is `web` split into? (`web.rdd.getNumPartitions()`)

### 4. **The tempting way — a Python UDF.** Write a Python function that labels `duration_ms` as `'fast'` (< 1000), `'medium'` (< 5000) or `'slow'`, wrap it with `F.udf`, and apply it. Show 5 rows. ⚠️ This works, but every row is shipped to a Python worker and Catalyst sees a black box.

### 5. **The Spark way — native built-ins.** Reproduce the exact same labelling with `F.when(...).otherwise(...)`. This stays inside the JVM, is vectorized, and Catalyst can optimize it. Show 5 rows.

### 6. Confirm both approaches agree: count rows **per `speed`** using the native version.

### 7. **The middle ground — a vectorized `pandas_udf`.** When you truly need Python, a `pandas_udf` processes a whole batch as a pandas Series (using Arrow), far faster than a row-by-row UDF. Write one that converts `duration_ms` to seconds and apply it. Show 5 rows.

### 8. **`repartition`** reshuffles the data into a chosen number of partitions (a full shuffle). Repartition `web` into 16 partitions and confirm the new count.

### 9. **`coalesce`** only *reduces* partitions, and avoids a full shuffle — ideal right before writing output. Coalesce `web` down to 2 partitions and confirm.

### 10. A realistic aggregation: **average `duration_ms` and event count per `page`**, slowest pages first (round the average).

### 11. If you will reuse a filtered DataFrame several times, **`cache`** it so Spark does not recompute it each time. Cache the `submit` events, materialize with `count()`, then check `storageLevel`.

### 12. **`count()` scans everything; `take()` does not.** `count()` the whole table (an action over all 250k rows), then `take(3)` — which stops as soon as it has 3 rows. Notice `take` returns a small Python list.